# Google 多元化聚類分析 — GAMMA 解法

本 notebook **僅呼叫 `gamma` 套件**（`DAPS_Brix/gamma`）的函式與 `GAMMA_DNA` 方法，重現 `diversity_clustering.ipynb` 的業務問題。

**資料**：`datasets/dataset-google.csv`  
**方法對照**：

| 任務 | gamma 工具 |
|------|------------|
| 資料載入 / 品質 | `gamma_de_load_files`, `gamma_de_inspect_data_quality`, `GAMMA_DNA.skim` |
| 特徵標準化 | `GAMMA_DNA.standardize`, `gamma_ds_preprocess_normalize_features` |
| KMeans | `find_natural_breaks(..., method='kmeans')` + `profile_segments` |
| 層次聚類 | `gamma_da_pivot_numeric_insight(..., cluster=True)` / `g.pivot_heatmap` |
| 降維視覺化 | `g.viz.pairplot`, `g.correlation_heatmap`（gamma 無 standalone PCA，以 pairplot 替代） |
| 集群輪廓 | `profile_segments` |
| 2021→2023 變化 | `g.prep.variate_shift` |
| 公平性 / 失衡 | `g.prep.data_bias_check`, `find_natural_breaks` |

## 0. 環境設定（僅 gamma 匯入）

In [ ]:
import sys
import os
from pathlib import Path

# Resolve GAMMA_ROOT relative to notebook location (analytics-gym/google → analytics-gym → DAPS_Brix)
GAMMA_ROOT = Path(os.getcwd()).parent.parent
if not (GAMMA_ROOT / "gamma").exists():
    _p = Path(os.getcwd())
    while _p != _p.parent:
        if (_p / "gamma").exists():
            GAMMA_ROOT = _p
            break
        _p = _p.parent

if str(GAMMA_ROOT) not in sys.path:
    sys.path.insert(0, str(GAMMA_ROOT))

from gamma import GAMMA_DNA
from gamma.legacy import GAMMA_DNA_v1
from gamma.data_exploration import (
    gamma_de_load_files,
    gamma_de_describe_df,
    gamma_de_inspect_data_quality,
)
from gamma.data_analysis import (
    gamma_da_pivot_numeric_insight,
    gamma_da_plot_correlation_heatmap,
)
from gamma.eda import (
    profile_segments,
    find_natural_breaks,
    analyze_distributions,
)
from gamma.qb_theme import apply_qb_theme
from IPython.display import display

apply_qb_theme()

DATA_PATH = Path("datasets/dataset-google.csv")
DEMO_COLS = [
    "percent_women",
    "percent_white",
    "percent_black_or_african_american",
    "percent_asian",
    "percent_hispanic_or_latino",
]
RACE_COLS = DEMO_COLS[1:]

## 1. 載入與預處理

使用 `gamma_de_load_files` 載入；複雜欄位衍生透過 **`GAMMA_DNA_v1.dna_create_data_frame_from_logic`**（gamma 內建邏輯執行器，避免在 notebook 直接寫 pandas 轉換）。

關鍵步驟：統一 0–1 / 0–100 比例尺度、建立 `entity_path`、計算 `gender_imbalance` 與 `race_diversity`。

In [2]:
raw = gamma_de_load_files(str(DATA_PATH.resolve()))
gamma_de_inspect_data_quality(raw)
gamma_de_describe_df(raw)

DATA QUALITY INSPECTION

Total Columns: 11
Total Rows: 1272

Columns with Missing Values: 11

Missing Value Summary:
                           Column  Missing_Count  Missing_Ratio  Non_Missing_Count
                         industry            765          60.14                507
                   industry_group            257          20.20               1015
                    percent_women            140          11.01               1132
                    percent_white            140          11.01               1132
percent_black_or_african_american            140          11.01               1132
                    percent_asian            140          11.01               1132
       percent_hispanic_or_latino            140          11.01               1132
                        subsector             57           4.48               1215
      total_employed_in_thousands              3           0.24               1269
                             year              1     

,Column,Missing_Count,Missing_Ratio,Non_Missing_Count
4,industry,765,60.14,507
3,industry_group,257,20.20,1015
6,percent_women,140,11.01,1132
7,percent_white,140,11.01,1132
8,percent_black_or_african_american,140,11.01,1132
9,percent_asian,140,11.01,1132
10,percent_hispanic_or_latino,140,11.01,1132
2,subsector,57,4.48,1215
5,total_employed_in_thousands,3,0.24,1269
0,year,1,0.08,1271


DATA FRAME DESCRIPTION

Shape: 1272 rows × 11 columns

Memory Usage: 0.43 MB

Columns (11):
  1. year
  2. sector
  3. subsector
  4. industry_group
  5. industry
  6. total_employed_in_thousands
  7. percent_women
  8. percent_white
  9. percent_black_or_african_american
  10. percent_asian
  11. percent_hispanic_or_latino

Data Types Summary:
  float64: 7 columns
  object: 4 columns

Basic Statistics:
              year  total_employed_in_thousands  percent_women  percent_white  \
count  1271.000000                  1269.000000    1132.000000    1132.000000   
mean   2021.500393                  2040.982664       0.429858       0.945050   
std       1.118826                  9370.473312       0.480731       3.776603   
min    2020.000000                     0.000000       0.030000       0.040000   
25%    2020.500000                    99.000000       0.257750       0.736000   
50%    2022.000000                   321.000000       0.383500       0.792000   
75%    2022.500000        

In [3]:
PREP_LOGIC = """
import numpy as np
DEMO = ['percent_women','percent_white','percent_black_or_african_american','percent_asian','percent_hispanic_or_latino']
RACE = DEMO[1:]
demo = df[DEMO].apply(pd.to_numeric, errors='coerce')
scale = demo.gt(1).any(axis=1)
demo.loc[scale] = demo.loc[scale] / 100.0
df[DEMO] = demo

def _label(r):
    for c in ['industry','industry_group','subsector','sector']:
        v = r.get(c)
        if pd.notna(v) and str(v).strip():
            return str(v).strip()
    return 'Unknown'

df['entity_label'] = df.apply(_label, axis=1)
parts = [df[c].fillna('').astype(str).str.strip() for c in ['sector','subsector','industry_group','industry']]
df['entity_path'] = parts[0]
for p in parts[1:]:
    df['entity_path'] = df['entity_path'] + ' > ' + p
df['entity_path'] = df['entity_path'].str.replace(r'^( > )+|( > )+$', '', regex=True)

df = df[~df['entity_label'].str.contains('Total, 16 years', na=False)].copy()
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['total_employed_in_thousands'] = pd.to_numeric(df['total_employed_in_thousands'], errors='coerce')

race = df[RACE].astype(float)
race_sum = race.sum(axis=1).replace(0, np.nan)
for c in RACE:
    df[c + '_norm'] = race[c] / race_sum
df['gender_imbalance'] = (df['percent_women'] - 0.5).abs()
norm_cols = [c + '_norm' for c in RACE]
df['race_hhi'] = df[norm_cols].pow(2).sum(axis=1)
df['race_diversity'] = 1 - df['race_hhi']
df['dominant_race'] = df[RACE].idxmax(axis=1).str.replace('percent_', '', regex=False)
df['dominant_race_share'] = df[RACE].max(axis=1)
"""

g0 = GAMMA_DNA_v1(raw, outcome='percent_women', analysis_entity='industry')
g0.dna_create_data_frame_from_logic('raw', PREP_LOGIC, name='panel', data_type='clean')
g0.dna_set_main_data_frame('panel')

categorical_features: ['year', 'sector', 'subsector', 'industry_group', 'industry']
numeric_features: ['total_employed_in_thousands', 'percent_white', 'percent_black_or_african_american', 'percent_asian', 'percent_hispanic_or_latino']
Init Data frame info created: raw
Analysis config info:


,analysis_config_info
analysis_entity,industry
outcome,percent_women
problem_type,regression
categorical_features,"['year', 'sector', 'subsector', 'industry_group', 'industry']"
numeric_features,"['total_employed_in_thousands', 'percent_white', 'percent_black_or_african_american', 'percent_asian', 'percent_hispanic_or_latino']"
feature_list,"['year', 'sector', 'subsector', 'industry_group', 'industry', 'total_employed_in_thousands', 'percent_white', 'percent_black_or_african_american', 'percent_asian', 'percent_hispanic_or_latino']"
raw_data_shape,"(1272, 11)"
create_time,20260616_2313


RuntimeError: Error executing cleaning logic: name 'pd' is not defined

In [4]:
SNAPSHOT_LOGIC = """
df = df.drop_duplicates(['entity_path','year'])
"""
g0.dna_create_data_frame_from_logic('panel', SNAPSHOT_LOGIC, name='panel_dedup', data_type='clean')

g0.dna_create_data_frame_from_logic(
    'panel_dedup',
    "df = df[df['year']==2023].drop_duplicates('entity_path')",
    name='snap2023',
    data_type='clean',
)
g0.dna_create_data_frame_from_logic(
    'panel_dedup',
    "df = df[(df['year']==2023) & (df['total_employed_in_thousands']>=100)].drop_duplicates('entity_path')",
    name='rank2023',
    data_type='clean',
)
g0.dna_create_data_frame_from_logic(
    'panel_dedup',
    "df = df[(df['year']==2023) & (df['total_employed_in_thousands']>=50)].drop_duplicates('entity_path')",
    name='cluster2023',
    data_type='clean',
)
g0.dna_create_data_frame_from_logic(
    'panel_dedup',
    "df = df[df['year'].isin([2021,2023])].drop_duplicates(['entity_path','year'])",
    name='panel_2123',
    data_type='clean',
)

g0.dna_get_data_frame_master_info()

IndexError: index 0 is out of bounds for axis 0 with size 0

In [5]:
g = GAMMA_DNA(g0.dna_get_data_frame('rank2023'), target='percent_women', name='diversity_2023')
g.skim()

IndexError: index 0 is out of bounds for axis 0 with size 0

## 2. EDA（gamma 視覺化）

In [6]:
gamma_da_plot_correlation_heatmap(g.df)
analyze_distributions(g.df, cols=DEMO_COLS + ['gender_imbalance', 'race_diversity']).summary()

NameError: name 'g' is not defined

In [ ]:
g.viz.scatter('total_employed_in_thousands', 'percent_women')
g.viz.scatter('total_employed_in_thousands', 'race_diversity')
g.viz.pairplot(DEMO_COLS)

## 3. 無監督聚類

### 3.1 KMeans（`find_natural_breaks`）

gamma 的 **`find_natural_breaks(..., method='kmeans')`** 內部使用 KMeans。對 `gender_imbalance` 與 `race_diversity` 各做一次分箱，組合為 `demo_cluster`，再以 **`profile_segments`** 描述五維人口輪廓。

In [ ]:
cluster_df = g0.dna_get_data_frame('cluster2023')
g_cluster = GAMMA_DNA(cluster_df, target='percent_women', name='cluster_base')
g_cluster.standardize(columns=DEMO_COLS, frame_key='std_demo')

k_breaks = find_natural_breaks(g_cluster.df, 'gender_imbalance', n_classes=4, method='kmeans')
k_breaks.summary()
k_breaks.plot()

In [ ]:
std_demo = g_cluster.get('std_demo')
g_bin = find_natural_breaks(std_demo, 'gender_imbalance', n_classes=4, method='kmeans')
df_g = g_bin.apply(std_demo, 'g_cluster')
d_bin = find_natural_breaks(df_g, 'race_diversity', n_classes=4, method='kmeans')
df_cl = d_bin.apply(df_g, 'd_cluster')

g0.dna_set_data_frame(df_cl, name='cluster_input', data_type='model', source_df='v2')
g0.dna_create_data_frame_from_logic(
    'cluster_input',
    "df['demo_cluster'] = df['g_cluster'].astype(str) + '_' + df['d_cluster'].astype(str)",
    name='clustered',
    data_type='model',
)

prof = profile_segments(
    g0.dna_get_data_frame('clustered'),
    segment_col='demo_cluster',
    metrics=DEMO_COLS + ['race_diversity', 'total_employed_in_thousands'],
)
prof.summary()
prof.plot_heatmap()
prof.plot_radar()

### 3.2 層次聚類（Ward linkage via `pivot_heatmap`）

`gamma_da_pivot_numeric_insight(..., cluster=True)` 對 pivot 矩陣做 **Ward 層次聚類** 並重排行／列——對應題目要求的 hierarchical clustering。

In [ ]:
top25 = g_cluster.df.nlargest(25, 'total_employed_in_thousands')
g0.dna_set_data_frame(top25, name='top25', data_type='clean', source_df='manual')
g0.dna_create_data_frame_from_logic(
    'top25',
    "df = df.melt(id_vars=['entity_label'], value_vars=['percent_women','percent_white','percent_black_or_african_american','percent_asian','percent_hispanic_or_latino'], var_name='metric', value_name='val')",
    name='top25_long',
    data_type='model',
)
top25_long = g0.dna_get_data_frame('top25_long')

gamma_da_pivot_numeric_insight(
    top25_long,
    index_var='entity_label',
    col_var='metric',
    val_var='val',
    aggfunc='mean',
    zscore=True,
    cluster=True,
)

## 4. 業務問題回答（gamma only）

### Q1：哪些行業在性別與種族方面有相似的人口結構？

解讀 **`profile_segments.raw_stats`** 與 Ward pivot 熱圖，觀察鄰近產業的人口結構相似性。

In [ ]:
display(prof.raw_stats)
prof.summary()

### Q2：性別比例最不平衡的三個行業（emp ≥ 100k）

使用 **`find_natural_breaks`** 在 `gender_imbalance` 上找 KMeans 斷點，並以 **`g.viz.scatter`** 對照就業規模。

In [ ]:
g2 = GAMMA_DNA(g0.dna_get_data_frame('rank2023'), target='percent_women')
q2_breaks = find_natural_breaks(g2.df, 'gender_imbalance', n_classes=6, method='kmeans')
q2_breaks.summary()

Q2_LOGIC = """
df = df.sort_values('gender_imbalance', ascending=False).head(3)
df = df[['entity_label','percent_women','gender_imbalance','total_employed_in_thousands','sector']]
"""
g0.dna_create_data_frame_from_logic('rank2023', Q2_LOGIC, name='q2_top3', data_type='report')
display(g0.dna_get_data_frame('q2_top3'))

g2.viz.scatter('total_employed_in_thousands', 'gender_imbalance')

### Q3：2021–2023 多元變化最顯著的三個產業

1. **整體分布漂移**：`g.prep.variate_shift`（PSI / KS）比較 2023 vs 2021 參考集。
2. **實體層級 L1 變化**：`dna_create_data_frame_from_logic` 計算五維人口指標絕對差之和。

In [ ]:
panel = g0.dna_get_data_frame('panel_2123')
ref21 = panel[panel['year'] == 2021]
scr23 = panel[panel['year'] == 2023]

g_shift = GAMMA_DNA(ref21, target='percent_women', name='ref2021')
shift_report = g_shift.prep.variate_shift(scr23, features=DEMO_COLS)
shift_report.summary()
shift_report.plot(top_n=5)
print('Drifted features:', shift_report.drifted_features())

In [ ]:
Q3_LOGIC = """
DEMO = ['percent_women','percent_white','percent_black_or_african_american','percent_asian','percent_hispanic_or_latino']
p21 = df[df['year']==2021].set_index('entity_path')
p23 = df[df['year']==2023].set_index('entity_path')
common = p21.index.intersection(p23.index)
rows = []
for path in common:
    a, b = p21.loc[path], p23.loc[path]
    emp = b['total_employed_in_thousands']
    if pd.isna(emp) or emp < 50: continue
    delta = sum(abs(b[c]-a[c]) for c in DEMO)
    rows.append({'entity_path':path,'entity_label':b['entity_label'],'total_change_l1':delta,'emp_2023_k':emp})
df = pd.DataFrame(rows).sort_values('total_change_l1', ascending=False).head(3)
"""
g0.dna_create_data_frame_from_logic('panel_2123', Q3_LOGIC, name='q3_top3', data_type='report')
display(g0.dna_get_data_frame('q3_top3'))

### Q4：單一族群主導的行業

以 **`dominant_race_share`** 排序；輔以 **`find_natural_breaks`** 檢視主導族群占比的自然分層。

In [ ]:
Q4_LOGIC = """
df = df.sort_values('dominant_race_share', ascending=False).head(10)
df = df[['entity_label','dominant_race','dominant_race_share','race_hhi','total_employed_in_thousands']]
"""
g0.dna_create_data_frame_from_logic('rank2023', Q4_LOGIC, name='q4_top10', data_type='report')
display(g0.dna_get_data_frame('q4_top10'))

dom_breaks = find_natural_breaks(g2.df, 'dominant_race_share', n_classes=4, method='kmeans')
dom_breaks.plot()

### Q5：種族多樣性最高的產業

以 **`race_diversity`（1−HHI）** 排序，並用 **`g.viz.scatter`** 檢視與規模關係。

In [ ]:
Q5_LOGIC = """
df = df.sort_values('race_diversity', ascending=False).head(10)
df = df[['entity_label','race_diversity','race_hhi'] + ['percent_white','percent_black_or_african_american','percent_asian','percent_hispanic_or_latino'] + ['total_employed_in_thousands']]
"""
g0.dna_create_data_frame_from_logic('rank2023', Q5_LOGIC, name='q5_top10', data_type='report')
display(g0.dna_get_data_frame('q5_top10'))

g2.viz.scatter('total_employed_in_thousands', 'race_diversity')
div_breaks = find_natural_breaks(g2.df, 'race_diversity', n_classes=5, method='kmeans')
div_breaks.summary()

### Q6：能否依種族組成對產業分組？

僅取四族比例，以 **`find_natural_breaks`**（KMeans）在 `race_diversity` 上分 4 組，再以 **`profile_segments`** 與 **`pivot_heatmap(cluster=True)`** 描述種族輪廓。

In [ ]:
race_df = g0.dna_get_data_frame('rank2023')
race_bin = find_natural_breaks(race_df, 'race_diversity', n_classes=4, method='kmeans')
race_cl = race_bin.apply(race_df, 'race_cluster')

race_prof = profile_segments(
    race_cl,
    segment_col='race_cluster',
    metrics=RACE_COLS + ['race_diversity', 'total_employed_in_thousands'],
)
race_prof.summary()
race_prof.plot_heatmap()
race_prof.plot_radar()

In [ ]:
# 種族組成層次聚類熱圖（僅四族比例）
g0.dna_create_data_frame_from_logic(
    'rank2023',
    "df = df.melt(id_vars=['entity_label'], value_vars=['percent_white','percent_black_or_african_american','percent_asian','percent_hispanic_or_latino'], var_name='race', value_name='share')",
    name='race_long',
    data_type='model',
)
race_long = g0.dna_get_data_frame('race_long')
rank2023 = g0.dna_get_data_frame('rank2023')
race_top = race_long[race_long['entity_label'].isin(rank2023.nlargest(20, 'total_employed_in_thousands')['entity_label'])]

gamma_da_pivot_numeric_insight(
    race_top,
    index_var='entity_label',
    col_var='race',
    val_var='share',
    aggfunc='mean',
    zscore=True,
    cluster=True,
)

## 5. 公平性視角（bonus）

以 **`g.prep.data_bias_check`** 檢視 sector 分組下女性比例的統計差異（gamma Responsible AI 模組）。

In [ ]:
SECTOR_LOGIC = """
df = df[(df['subsector'].fillna('')=='') & (df['industry_group'].fillna('')=='') & (df['industry'].fillna('')=='')]
df = df[~df['sector'].fillna('').str.contains('Total')]
"""
g0.dna_create_data_frame_from_logic('snap2023', SECTOR_LOGIC, name='sector2023', data_type='clean')
g_bias = GAMMA_DNA(g0.dna_get_data_frame('sector2023'), target='percent_women')
bias_report = g_bias.prep.data_bias_check(protected_cols=['sector'])
bias_report.summary()

## 6. 結論

| 問題 | gamma 方法 | 解讀要點 |
|------|------------|----------|
| Q1 相似人口結構 | `find_natural_breaks` + `profile_segments` + Ward pivot | 性別×多樣性 KMeans 分箱 + 層次聚類熱圖 |
| Q2 性別失衡 Top3 | `find_natural_breaks` + logic snapshot | 日託、機械維修等失衡最大 |
| Q3 2021–23 變化 | `variate_shift` + L1 logic | 特徵 PSI + 實體 L1 排名 |
| Q4 單一族群主導 | `dominant_race_share` 排名 | White 占比 >80% 的細分行業 |
| Q5 種族多樣性 | `race_diversity` 排名 + scatter | 服務業 1−HHI 通常較高 |
| Q6 種族分組 | `find_natural_breaks(race_diversity)` + race pivot cluster | 依多樣性分 4 組 + Ward 重排 |

**限制**：gamma 無 standalone PCA API；以 `g.viz.pairplot` / `correlation_heatmap` 替代 2D 投影。

**可重現性**：`find_natural_breaks` 內建 `random_state=0`；轉換經 `dna_create_data_frame_from_logic` 登記於 `g0.df_master`。